# ET Prime Pitch — Fine-Tune Qwen2.5-7B
**GPU: A100 + High RAM** | Just click Runtime → Run All

In [ ]:
import os, IPython
IPython.display.display(IPython.display.Javascript("function C(){document.querySelector(\"colab-connect-button\").click()}setInterval(C,60000)"))
print("Setup done.")

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers
!pip install -q datasets
print("Deps installed.")

In [ ]:
import subprocess, os, json
REPO = "Mohit1053/et-prime-finetune"
for f in ["train.jsonl", "eval.jsonl"]:
    if not os.path.exists(f"/content/{f}"):
        print(f"Downloading {f}...")
        subprocess.run(["wget","-q","-O",f"/content/{f}",f"https://github.com/{REPO}/releases/download/v1.0/{f}"],check=True)
tc=sum(1 for _ in open("/content/train.jsonl"))
ec=sum(1 for _ in open("/content/eval.jsonl"))
print(f"Train: {tc}, Eval: {ec}")

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct", max_seq_length=4096,
    dtype=None, load_in_4bit=True)
print(f"Model loaded: {model.num_parameters():,} params")

In [ ]:
model = FastLanguageModel.get_peft_model(model, r=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=64, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42)
tr=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {tr:,} params")

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train":"/content/train.jsonl","eval":"/content/eval.jsonl"})
def fmt(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)}
ds = ds.map(fmt, num_proc=4)
print(f"Train: {len(ds["train"])}, Eval: {len(ds["eval"])}")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import time
trainer = SFTTrainer(model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["eval"],
    dataset_text_field="text", max_seq_length=4096, dataset_num_proc=4, packing=True,
    args=TrainingArguments(output_dir="/content/ckpt", num_train_epochs=2,
        per_device_train_batch_size=4, gradient_accumulation_steps=8,
        learning_rate=2e-5, lr_scheduler_type="cosine", warmup_ratio=0.05,
        weight_decay=0.01, fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(), logging_steps=25,
        eval_strategy="steps", eval_steps=250, save_strategy="steps",
        save_steps=500, save_total_limit=3, optim="adamw_8bit",
        seed=42, report_to="none"))
start=time.time()
print("Training...")
stats=trainer.train()
print(f"Done! {(time.time()-start)/3600:.1f}h, loss: {stats.training_loss:.4f}")

In [ ]:
ev=trainer.evaluate()
print(f"Eval loss: {ev["eval_loss"]:.4f}, perplexity: {2**ev["eval_loss"]:.2f}")

In [ ]:
import json
FastLanguageModel.for_inference(model)
with open("/content/eval.jsonl") as f: test=json.loads(f.readline())
inp=tokenizer.apply_chat_template(test["messages"][:2],tokenize=True,add_generation_prompt=True,return_tensors="pt").to("cuda")
out=model.generate(input_ids=inp,max_new_tokens=512,temperature=0.9,do_sample=True)
resp=tokenizer.decode(out[0][inp.shape[-1]:],skip_special_tokens=True)
print("GENERATED:"); print(resp)
print("
EXPECTED:"); print(test["messages"][2]["content"])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
SAVE="/content/drive/MyDrive/ai-toolkit/models/et-prime-pitch/merged"
os.makedirs(SAVE,exist_ok=True)
print("Saving merged model to Drive...")
model.save_pretrained_merged(SAVE, tokenizer, save_method="merged_16bit")
print(f"Saved: {SAVE}")
print("
" + "="*60)
print("DONE! Now open serve.ipynb to start inference server.")
print("="*60)
import requests
try: requests.post("https://ntfy.sh/mohit-colab",data=f"Fine-tuning done! Loss: {stats.training_loss:.4f}",headers={"Title":"FT Complete","Priority":"high"})
except: pass